In [ ]:
!pip install sahi ultralytics supervision -q
print("installed")

In [ ]:
import torch
assert torch.cuda.is_available()
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
import supervision as sv
import inspect

sig = inspect.signature(sv.ByteTrack.__init__)
print("Default parameter values ByteTrack:")
for name, param in sig.parameters.items():
    if param.default is not inspect.Parameter.empty:
        print(f"  {name} = {param.default}")

print("\n" + sv.ByteTrack.__doc__[:2000] if sv.ByteTrack.__doc__ else "No docstring")

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Checking the frame size to see if it matches what was used during training
sample_img = cv2.imread(str(frame_files[100]))
print(f"Size of the new frame: {sample_img.shape[1]}x{sample_img.shape[0]}")

direct_model = YOLO(MODEL_PATH)
sample_idx = [0, len(frame_files)//5, 2*len(frame_files)//5,
              3*len(frame_files)//5, 4*len(frame_files)//5, len(frame_files)-1]

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
for ax, idx in zip(axes.flat, sample_idx):
    fp = frame_files[idx]
    res = direct_model(str(fp), imgsz=IMG_SIZE, conf=CONF_THRESH, verbose=False, device="cuda")
    img = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
    n = len(res[0].boxes)
    for box in res[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    ax.imshow(img)
    ax.set_title(f"Frame {idx}: {n} chicken found", fontsize=11)
    ax.axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/check.jpg", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
from pathlib import Path

# Path to the dataset containing 25,200 dense frames
FRAMES_DIR    = "/kaggle/input/datasets/natair/chickens-drone-25k/chickens_drone_25k"

# Trained model
MODEL_PATH    = "/kaggle/input/models/natair/chicken-yolo8m-boosted/pytorch/default/1/best.pt"

OUTPUT_DIR    = "/kaggle/working"
LABEL_NAME    = "chicken"

# Detection parameters
IMG_SIZE      = 1280
CONF_THRESH   = 0.20
SLICE_SIZE    = 640
OVERLAP_RATIO = 0.2

CHECKPOINT_EVERY = 2000     # save progress every N frames

RESUME_CHECKPOINT = None
#RESUME_CHECKPOINT = "/kaggle/input/models/natair/detections-checkpoint-2/pytorch/default/1/detections_checkpoint.pkl"

# Parameters ByteTrack
VIDEO_FPS                    = 60     # fps
TRACK_ACTIVATION_THRESHOLD   = 0.5
LOST_TRACK_BUFFER            = 30     # 60 frames at 60 fps = 1 second of dropout tolerance
MINIMUM_MATCHING_THRESHOLD   = 0.3

In [ ]:
!nvidia-smi
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))

In [ ]:
import pickle
import numpy as np
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from tqdm import tqdm
import time

EXTS = {'.jpg', '.jpeg', '.png'}
frame_files = sorted(
    [p for p in Path(FRAMES_DIR).rglob("*") if p.suffix.lower() in EXTS],
    key=lambda p: p.stem
)

N = len(frame_files)
print(f" Found frames: {N}")
assert N > 0, f"Frames weren't found in {FRAMES_DIR}"

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=MODEL_PATH,
    confidence_threshold=CONF_THRESH,
    device="cuda",
)
print(" Model downloaded in SAHI")

print("\nEstimate of the time required for a full run: ")
t0 = time.time()
for fp in frame_files[:20]:
    get_sliced_prediction(
        str(fp), detection_model,
        slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
        overlap_height_ratio=OVERLAP_RATIO, overlap_width_ratio=OVERLAP_RATIO,
        verbose=0,
    )
per_frame_sec = (time.time() - t0) / 20
est_total_min = per_frame_sec * N / 60
print(f"  ~{per_frame_sec*1000:.0f} ms/frame;  all dataset = {est_total_min:.1f} minutes")

In [ ]:
import re

def frame_number(path):
    match = re.search(r'(\d+)', path.stem)
    return int(match.group(1)) if match else 0

frame_files = sorted(frame_files, key=frame_number)

In [ ]:
from ultralytics import YOLO
direct_model = YOLO(MODEL_PATH)

BATCH_SIZE = 16

all_detections = {}
frame_paths_str = [str(fp) for fp in frame_files]

if RESUME_CHECKPOINT and Path(RESUME_CHECKPOINT).exists():
    with open(RESUME_CHECKPOINT, "rb") as f:
        saved = pickle.load(f)
    all_detections = saved["detections"]
    start_idx = saved["last_idx"] + 1
    print(f" Continue with frame {start_idx} (already processed {len(all_detections)})")
else:
    print(" Beginning from scratch")

checkpoint_path = Path(OUTPUT_DIR) / "detections_checkpoint.pkl"

for batch_start in tqdm(range(0, N, BATCH_SIZE), desc="YOLO detecction"):
    batch_paths = frame_paths_str[batch_start:batch_start + BATCH_SIZE]
    results = direct_model(
        batch_paths, imgsz=IMG_SIZE, conf=CONF_THRESH,
        iou=0.35,
        verbose=False, device="cuda",
    )
    for i, res in enumerate(results):
        idx = batch_start + i
        boxes = res.boxes.xyxy.cpu().numpy()
        confs = res.boxes.conf.cpu().numpy()
        if len(boxes) > 0:
            all_detections[idx] = np.hstack([boxes, confs.reshape(-1, 1)])
        else:
            all_detections[idx] = np.zeros((0, 5))

    if (batch_start + BATCH_SIZE) % 2000 < BATCH_SIZE:
        with open(checkpoint_path, "wb") as f:
            pickle.dump({"detections": all_detections, "last_idx": batch_start + len(batch_paths) - 1}, f)

with open(checkpoint_path, "wb") as f:
    pickle.dump({"detections": all_detections, "last_idx": N - 1}, f)

total_det = sum(len(v) for v in all_detections.values())
print(f"\n Detection completed: {len(all_detections)}/{N} frames")
print(f" Overall detections: {total_det}  |  Average: {total_det/N:.1f} per frame")

# Checking frames
counts = [len(v) for v in all_detections.values()]
print(f"  Median number of detections per frame: {np.median(counts):.0f}")
print(f"  Maximum per frame: {max(counts)}")
print(f"  Frames with 0 detections: {sum(1 for c in counts if c == 0)}")

In [ ]:
all_detections = {}   # {frame_idx: np.array([[x1,y1,x2,y2,conf], ...])}
start_idx = 0

if RESUME_CHECKPOINT and Path(RESUME_CHECKPOINT).exists():
    with open(RESUME_CHECKPOINT, "rb") as f:
        saved = pickle.load(f)
    all_detections = saved["detections"]
    start_idx = saved["last_idx"] + 1
    print(f" Continue with frame {start_idx} (already processed {len(all_detections)})")
else:
    print(" Beginning from scratch")

checkpoint_path = Path(OUTPUT_DIR) / "detections_checkpoint.pkl"

for idx in tqdm(range(start_idx, N), desc="SAHI detection", initial=start_idx, total=N):
    fp = frame_files[idx]
    result = get_sliced_prediction(
        str(fp), detection_model,
        slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
        overlap_height_ratio=OVERLAP_RATIO, overlap_width_ratio=OVERLAP_RATIO,
        verbose=0,
    )
    boxes = [
        [o.bbox.minx, o.bbox.miny, o.bbox.maxx, o.bbox.maxy, o.score.value]
        for o in result.object_prediction_list
    ]
    all_detections[idx] = np.array(boxes) if boxes else np.zeros((0, 5))

    if (idx + 1) % CHECKPOINT_EVERY == 0:
        with open(checkpoint_path, "wb") as f:
            pickle.dump({"detections": all_detections, "last_idx": idx}, f)

# Final saving
with open(checkpoint_path, "wb") as f:
    pickle.dump({"detections": all_detections, "last_idx": N - 1}, f)

total_det = sum(len(v) for v in all_detections.values())
print(f"\n Detection completed: {len(all_detections)}/{N} frames")
print(f" Overall detections: {total_det}  |  On average: {total_det/N:.1f} per frame")

In [ ]:
# Restoration of results
!pip install sahi ultralytics supervision -q

import torch
from pathlib import Path
from ultralytics import YOLO
from sahi import AutoDetectionModel

assert torch.cuda.is_available()
print(f"GPU: {torch.cuda.get_device_name(0)}")

FRAMES_DIR    = "/kaggle/input/datasets/natair/chickens-drone-25k/chickens_drone_25k"
MODEL_PATH    = "/kaggle/input/models/natair/chicken-yolo8m-boosted/pytorch/default/1/best.pt"
OUTPUT_DIR    = "/kaggle/working"
LABEL_NAME    = "chicken"

IMG_SIZE      = 1280
CONF_THRESH   = 0.25
SLICE_SIZE    = 640
OVERLAP_RATIO = 0.2
VIDEO_FPS     = 60

EXTS = {'.jpg', '.jpeg', '.png'}
frame_files = sorted(
    [p for p in Path(FRAMES_DIR).rglob("*") if p.suffix.lower() in EXTS],
    key=lambda p: p.stem
)
N = len(frame_files)
print(f" Frames found: {N}")

direct_model = YOLO(MODEL_PATH)

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=MODEL_PATH,
    confidence_threshold=CONF_THRESH,
    device="cuda",
)

In [ ]:
# Checking if lowering the IoU threshold in NMS help

import matplotlib.pyplot as plt
import cv2

cluster_frame_indices = [981, 1146, 1386, 2004]

fig, axes = plt.subplots(2, len(cluster_frame_indices), figsize=(6*len(cluster_frame_indices), 10))

for col, idx in enumerate(cluster_frame_indices):
    fp = frame_files[idx]
    img_orig = cv2.imread(str(fp))

    res_old = direct_model(str(fp), imgsz=IMG_SIZE, conf=CONF_THRESH, iou=0.7, verbose=False, device="cuda")
    img_old = cv2.cvtColor(img_orig.copy(), cv2.COLOR_BGR2RGB)
    for box in res_old[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img_old, (x1, y1), (x2, y2), (255, 0, 0), 2)
    axes[0, col].imshow(img_old)
    axes[0, col].set_title(f"Frame {idx}: IoU=0.7 (was): {len(res_old[0].boxes)} chicken", fontsize=10)
    axes[0, col].axis("off")

    res_new = direct_model(str(fp), imgsz=IMG_SIZE, conf=CONF_THRESH, iou=0.35, verbose=False, device="cuda")
    img_new = cv2.cvtColor(img_orig.copy(), cv2.COLOR_BGR2RGB)
    for box in res_new[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img_new, (x1, y1), (x2, y2), (0, 255, 0), 2)
    axes[1, col].imshow(img_new)
    axes[1, col].set_title(f"Frame {idx}: IoU=0.35 (new): {len(res_new[0].boxes)} chicken", fontsize=10)
    axes[1, col].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/nms_comparison.jpg", dpi=100, bbox_inches="tight")
plt.show()

# Checking SAHI vs direct inference specifically on heaps

from sahi.predict import get_sliced_prediction

fig, axes = plt.subplots(2, len(cluster_frame_indices), figsize=(6*len(cluster_frame_indices), 10))

for col, idx in enumerate(cluster_frame_indices):
    fp = frame_files[idx]
    img_orig = cv2.imread(str(fp))

    # Direct inference (current approach)
    res_direct = direct_model(str(fp), imgsz=IMG_SIZE, conf=CONF_THRESH, verbose=False, device="cuda")
    img_d = cv2.cvtColor(img_orig.copy(), cv2.COLOR_BGR2RGB)
    for box in res_direct[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img_d, (x1, y1), (x2, y2), (255, 0, 0), 2)
    axes[0, col].imshow(img_d)
    axes[0, col].set_title(f"Frame {idx}: direct: {len(res_direct[0].boxes)} chicken", fontsize=10)
    axes[0, col].axis("off")

    # SAHI (preserves resolution in tiles)
    res_sahi = get_sliced_prediction(
        str(fp), detection_model,
        slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
        overlap_height_ratio=OVERLAP_RATIO, overlap_width_ratio=OVERLAP_RATIO,
        verbose=0,
    )
    img_s = cv2.cvtColor(img_orig.copy(), cv2.COLOR_BGR2RGB)
    for o in res_sahi.object_prediction_list:
        x1, y1, x2, y2 = int(o.bbox.minx), int(o.bbox.miny), int(o.bbox.maxx), int(o.bbox.maxy)
        cv2.rectangle(img_s, (x1, y1), (x2, y2), (0, 255, 0), 2)
    axes[1, col].imshow(img_s)
    axes[1, col].set_title(f"Frame {idx}: SAHI: {len(res_sahi.object_prediction_list)} chicken", fontsize=10)
    axes[1, col].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/sahi_vs_direct_clusters.jpg", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
import pickle
import numpy as np
from pathlib import Path
from tqdm import tqdm

USE_CACHED_RESULTS = False
CACHED_RESULTS_PATH = "/kaggle/input/chicken-detections-cache/all_detections_full.pkl"


if USE_CACHED_RESULTS:
    assert Path(CACHED_RESULTS_PATH).exists(), (
        f"Cache not found: {CACHED_RESULTS_PATH}"
    )
    with open(CACHED_RESULTS_PATH, "rb") as f:
        all_detections = pickle.load(f)

    assert len(all_detections) == N, (
        f"In cache {len(all_detections)} frames, and the dataset contains {N} frames"
    )

    total_det = sum(len(v) for v in all_detections.values())
    print(f" Loaded from cache: {len(all_detections)} frames, {total_det} detections")

else:
    from ultralytics import YOLO

    direct_model = YOLO(MODEL_PATH)
    BATCH_SIZE = 16
    frame_paths_str = [str(fp) for fp in frame_files]

    checkpoint_path = Path(OUTPUT_DIR) / "detections_checkpoint.pkl"

    RESUME_WITHIN_SESSION = False

    all_detections = {}
    start_batch = 0

    if RESUME_WITHIN_SESSION and checkpoint_path.exists():
        with open(checkpoint_path, "rb") as f:
            saved = pickle.load(f)
        all_detections = saved["detections"]
        last_idx = saved["last_idx"]
        start_batch = ((last_idx + 1) // BATCH_SIZE) * BATCH_SIZE
        print(f"Summarize with frame {start_batch} (there are already {len(all_detections)} frames in the cache)")
    else:
        print(" Counting from scratch")

    for batch_start in tqdm(range(start_batch, N, BATCH_SIZE), desc="YOLO detection"):
        batch_paths = frame_paths_str[batch_start:batch_start + BATCH_SIZE]
        results = direct_model(
            batch_paths, imgsz=IMG_SIZE, conf=CONF_THRESH,
            iou=0.35, verbose=False, device="cuda",
        )
        for i, res in enumerate(results):
            idx = batch_start + i
            boxes = res.boxes.xyxy.cpu().numpy()
            confs = res.boxes.conf.cpu().numpy()
            all_detections[idx] = (
                np.hstack([boxes, confs.reshape(-1, 1)]) if len(boxes) > 0 else np.zeros((0, 5))
            )

        if (batch_start + BATCH_SIZE) % 2000 < BATCH_SIZE:
            with open(checkpoint_path, "wb") as f:
                pickle.dump({"detections": all_detections, "last_idx": batch_start + len(batch_paths) - 1}, f)

    total_det = sum(len(v) for v in all_detections.values())
    print(f"\n Detection completed: {len(all_detections)}/{N} frames, {total_det} detections")

    final_path = Path(OUTPUT_DIR) / "all_detections_full.pkl"
    with open(final_path, "wb") as f:
        pickle.dump(all_detections, f)

counts = [len(v) for v in all_detections.values()]
print(f"\n Median number of detections per frame: {np.median(counts):.0f}  |  Max: {max(counts)}")

In [ ]:
import supervision as sv

tracker = sv.ByteTrack(
    frame_rate=60,
    minimum_consecutive_frames=3,
    lost_track_buffer=65,
)

tracks_by_frame = {}   # {frame_idx: {track_id: [x1,y1,x2,y2]}}

for idx in tqdm(range(N), desc="ByteTrack"):
    dets = all_detections.get(idx, np.zeros((0, 5)))

    if len(dets) > 0:
        detections = sv.Detections(
            xyxy=dets[:, :4],
            confidence=dets[:, 4],
            class_id=np.zeros(len(dets), dtype=int),
        )
    else:
        detections = sv.Detections.empty()

    tracked = tracker.update_with_detections(detections)

    tracks_by_frame[idx] = {}
    for box, tid in zip(tracked.xyxy, tracked.tracker_id):
        tracks_by_frame[idx][int(tid)] = box.tolist()

all_track_ids = sorted({tid for fd in tracks_by_frame.values() for tid in fd})
print(f" Tracking complete")
print(f" Unique tracks: {len(all_track_ids)}")

In [ ]:
track_lengths = {}
for tid in all_track_ids:
    frames_present = [f for f, fd in tracks_by_frame.items() if tid in fd]
    track_lengths[tid] = len(frames_present)

lengths = list(track_lengths.values())
print(f"Track length (in frames, at 60 fps):")
print(f"Median: {np.median(lengths):.0f} frames (~{np.median(lengths)/60:.1f} sec)")
print(f"Average: {np.mean(lengths):.1f} frames")

In [ ]:
# The Hungarian Algorithm
import numpy as np
from scipy.optimize import linear_sum_assignment

displacements = []
for tid in all_track_ids:
    frames_present = sorted(f for f, fd in tracks_by_frame.items() if tid in fd)
    for i in range(1, len(frames_present)):
        f_prev, f_cur = frames_present[i-1], frames_present[i]
        if f_cur - f_prev != 1:
            continue
        b_prev = tracks_by_frame[f_prev][tid]
        b_cur  = tracks_by_frame[f_cur][tid]
        p_prev = ((b_prev[0]+b_prev[2])/2, (b_prev[1]+b_prev[3])/2)
        p_cur  = ((b_cur[0]+b_cur[2])/2,  (b_cur[1]+b_cur[3])/2)
        displacements.append(np.hypot(p_cur[0]-p_prev[0], p_cur[1]-p_prev[1]))

if displacements:
    MAX_SPEED_PX_PER_FRAME = np.percentile(displacements, 95) * 1.3
else:
    MAX_SPEED_PX_PER_FRAME = 5.0

MAX_GAP_FRAMES = 90

print(f"Calibration Based on Data:")
print(f"  95th percentile of offset per frame: {np.percentile(displacements, 95):.1f}px")
print(f"  MAX_SPEED_PX_PER_FRAME = {MAX_SPEED_PX_PER_FRAME:.1f}px")
print(f"  MAX_GAP_FRAMES = {MAX_GAP_FRAMES} frames (~{MAX_GAP_FRAMES/VIDEO_FPS:.1f} sec)")

track_info = {}
for tid in all_track_ids:
    frames_present = sorted(f for f, fd in tracks_by_frame.items() if tid in fd)
    start_f, end_f = frames_present[0], frames_present[-1]
    sb = tracks_by_frame[start_f][tid]
    eb = tracks_by_frame[end_f][tid]
    track_info[tid] = {
        "start_f": start_f, "end_f": end_f,
        "start_pos": ((sb[0]+sb[2])/2, (sb[1]+sb[3])/2),
        "end_pos":   ((eb[0]+eb[2])/2, (eb[1]+eb[3])/2),
        "w": sb[2]-sb[0], "h": sb[3]-sb[1],
    }

tids = list(all_track_ids)
n = len(tids)
INF = 1e9
cost = np.full((n, n), INF)

for i, tid_a in enumerate(tids):
    ea, pa = track_info[tid_a]["end_f"], track_info[tid_a]["end_pos"]
    wa, ha = track_info[tid_a]["w"], track_info[tid_a]["h"]
    for j, tid_b in enumerate(tids):
        if tid_a == tid_b:
            continue
        sb_f = track_info[tid_b]["start_f"]
        gap = sb_f - ea
        if gap <= 0 or gap > MAX_GAP_FRAMES:
            continue

        pb = track_info[tid_b]["start_pos"]
        dist = np.hypot(pa[0]-pb[0], pa[1]-pb[1])
        max_allowed = MAX_SPEED_PX_PER_FRAME * gap
        if dist > max_allowed:
            continue

        wb, hb = track_info[tid_b]["w"], track_info[tid_b]["h"]
        size_ratio = max(wa, wb) / max(min(wa, wb), 1e-6)
        if size_ratio > 2.0:
            continue

        cost[i, j] = dist

row_ind, col_ind = linear_sum_assignment(cost)
links = [(tids[i], tids[j]) for i, j in zip(row_ind, col_ind) if cost[i, j] < INF]
print(f"\n Matches found: {len(links)}")

parent = {tid: tid for tid in tids}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for a, b in links:
    union(a, b)

merged_tracks_by_frame = {}
for f, fd in tracks_by_frame.items():
    merged_tracks_by_frame[f] = {}
    for tid, box in fd.items():
        new_id = find(tid)
        merged_tracks_by_frame[f][new_id] = box

new_track_ids = sorted(set(find(tid) for tid in tids))

print(f"  Before stitching:    {len(tids)} tracks")
print(f"  After stitching: {len(new_track_ids)} tracks")

tracks_by_frame = merged_tracks_by_frame
all_track_ids = new_track_ids

new_lengths = [len(sorted(f for f, fd in tracks_by_frame.items() if tid in fd)) for tid in all_track_ids]
print(f"\nAfter stitching:")
print(f"  Median length: {np.median(new_lengths):.0f} frames (~{np.median(new_lengths)/VIDEO_FPS:.1f} sec)")
print(f"  Short clips (<=10 frames): {sum(1 for l in new_lengths if l <= 10)} from {len(all_track_ids)}")

In [ ]:
import xml.etree.ElementTree as ET
from xml.dom import minidom

root = ET.Element("annotations")
ET.SubElement(root, "version").text = "1.1"
meta = ET.SubElement(root, "meta")
task = ET.SubElement(meta, "task")
ET.SubElement(task, "size").text = str(N)
lbls = ET.SubElement(task, "labels")
lbl  = ET.SubElement(lbls, "label")
ET.SubElement(lbl, "name").text = LABEL_NAME

for tid in all_track_ids:
    track_el = ET.SubElement(root, "track", {
        "id": str(tid), "label": LABEL_NAME, "source": "bytetrack_dense",
    })
    frames_present = sorted(f for f, fd in tracks_by_frame.items() if tid in fd)
    prev_f = None
    for f in frames_present:
        x1, y1, x2, y2 = tracks_by_frame[f][tid]
        if prev_f is not None and f - prev_f > 1:
            ET.SubElement(track_el, "box", {
                "frame": str(prev_f + 1),
                "xtl": "0", "ytl": "0", "xbr": "1", "ybr": "1",
                "outside": "1", "occluded": "0", "keyframe": "1",
            })
        ET.SubElement(track_el, "box", {
            "frame": str(f),
            "xtl": f"{x1:.1f}", "ytl": f"{y1:.1f}",
            "xbr": f"{x2:.1f}", "ybr": f"{y2:.1f}",
            "outside": "0", "occluded": "0", "keyframe": "1",
        })
        prev_f = f

xml_str = minidom.parseString(ET.tostring(root)).toprettyxml(indent="  ")
out_xml = Path(OUTPUT_DIR) / "tracks_dense.xml"
out_xml.write_text(xml_str, encoding="utf-8")
print(f" Tracks saved: {out_xml}")

In [ ]:
import cv2

def color_for_id(tid):
    np.random.seed(tid)
    return tuple(int(c) for c in np.random.randint(60, 255, 3))

preview_path = f"{OUTPUT_DIR}/tracking_preview.mp4"
max_preview_frames = min(1800, N)   # the first 30 seconds at 60fps

writer = None
for idx in range(max_preview_frames):
    frame = cv2.imread(str(frame_files[idx]))
    if writer is None:
        h, w = frame.shape[:2]
        writer = cv2.VideoWriter(preview_path, cv2.VideoWriter_fourcc(*"mp4v"), VIDEO_FPS, (w, h))

    for tid, box in tracks_by_frame.get(idx, {}).items():
        x1, y1, x2, y2 = map(int, box)
        c = color_for_id(tid)
        cv2.rectangle(frame, (x1, y1), (x2, y2), c, 2)
        cv2.putText(frame, f"#{tid}", (x1, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, c, 1)
    writer.write(frame)

if writer:
    writer.release()
print(f" Preview saved: {preview_path} (first {max_preview_frames} frames, ~{max_preview_frames/VIDEO_FPS:.0f} sec)")

In [ ]:
import shutil

bundle_dir = Path(OUTPUT_DIR) / "_bundle"
bundle_dir.mkdir(exist_ok=True)
shutil.copy(out_xml, bundle_dir / out_xml.name)
shutil.copy(preview_path, bundle_dir / "tracking_preview.mp4")
shutil.copy(checkpoint_path, bundle_dir / "detections_checkpoint.pkl")

archive_path = shutil.make_archive(f"{OUTPUT_DIR}/dense_tracking_results", "zip", bundle_dir)
shutil.rmtree(bundle_dir)